# 06 - Global Assignment (Plan Section 9)

`belief.py`'s cross-bus-date suppression (used live in the labeling UI)
is a single, non-iterated pairwise-discount heuristic: it can leave two
buses both claiming the same device if neither individually clears its
suppression threshold. This notebook replaces that heuristic, **per
date**, with a real joint solve: minimum-weight full bipartite matching
between buses and devices, so a device is matched to at most one bus
that day.

**Cost input**: `belief.compute_raw_beliefs`'s *pre-suppression*
posterior (`cost = -log(posterior)`), for the same reason the module
docstring gives -- this is what the suppression heuristic itself
approximates, so the global solve supersedes it rather than layering on
top of it.

**Sparse, not dense**: confirmed live (see `app/assignment.py`'s
docstring) a weekday's bipartite graph is ~1,600 buses x ~1,400 devices
at ~0.7% density, so `scipy.sparse.csgraph.min_weight_full_bipartite_matching`
(sparse LAPJVsp) is the right tool, not a dense
`scipy.optimize.linear_sum_assignment`.

**"None of these"**: every bus gets its own dummy "assign to none"
column (cost = that bus's own `NONE_OPTION` posterior from the same
belief computation), so a bus with no good candidate isn't forced onto
a bad one -- confirmed live, sample date assigned 397/1,609 buses to
"none".

**Zero-candidate bus-dates**: 1,325 of 41,332 valid bus-dates (3.2%,
concentrated in just 51 buses) have no candidate at all -- these never
enter the solver, and are written with `method='no_candidates'`
instead.

**Margin** (plan Section 9.3): for each assigned bus, re-solve the
*whole date* with that bus's edge forbidden and take the increase in
total cost. Confirmed live: ~2.3ms/bus, ~3.6s for a full weekday
including all ~1,600 margin re-solves -- cheap enough to do exactly as
specified, not an approximation.

**Batch job, not live**: per the plan and the labeling app's own
docstring, this runs as a periodic background job -- `belief.py`'s
cross-suppression stays as-is for the interactive labeling UI.


In [1]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))
sys.path.insert(0, str(_root / "ml" / "bus_matching_model" / "app"))

In [2]:
import datetime
import time

import assignment
import belief
import db
import numpy as np
import pandas as pd
import psycopg
import registry

from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Table

`(bus_id, date)` primary key, one row per valid bus-date (all 41,332,
not just the 40,007 with candidates) -- Section 13's "include an
explicit unresolved bucket" starts here: `method='no_candidates'` rows
are exactly that bucket for this layer.


In [3]:
conn.execute("DROP TABLE IF EXISTS ml.bus_matching_global_assignment;")
ddl = (
    "CREATE TABLE ml.bus_matching_global_assignment (\n"
    "    bus_id     text NOT NULL,\n"
    "    date       date NOT NULL,\n"
    "    device_id  text,\n"
    "    cost       double precision,\n"
    "    margin     double precision,\n"
    "    method     text NOT NULL,\n"
    "    PRIMARY KEY (bus_id, date)\n"
    ");"
)
conn.execute(ddl)
conn.commit()
print("ml.bus_matching_global_assignment created")

ml.bus_matching_global_assignment created


## Solve every date

Load candidates/votes/model once (same pattern as the labeling app),
compute the ramped `model_prior_weight` from the latest trained run,
then solve each date independently and `COPY` the results in.


In [4]:
run_row = db.fetch_latest_model_run(conn)
model_state = registry.load_run(run_row) if run_row else None
if model_state is not None:
    model_state["run_row"] = run_row
features = db.load_available_features()

candidates_all = db.compute_candidate_scores(conn, features, model_state)
votes_all = db.fetch_all_votes(conn)
model_prior_weight = belief.model_prior_weight_for(
    (run_row or {}).get("n_resolved_bus_dates"), (run_row or {}).get("test_ece")
)
run_id = (run_row or {}).get("run_id")
print(f"model_prior_weight={model_prior_weight:.3f}, run_id={run_id}")

with conn.cursor() as cur:
    cur.execute(
        "SELECT DISTINCT trip_date FROM ml.trip_validity_final "
        "WHERE is_valid ORDER BY trip_date;"
    )
    all_dates = [row[0] for row in cur.fetchall()]

with conn.cursor() as cur:
    cur.execute(
        "SELECT DISTINCT bus_id, trip_date FROM ml.trip_validity_final WHERE is_valid;"
    )
    all_bus_dates = pd.DataFrame(cur.fetchall(), columns=["bus_id", "date"])

print(f"{len(all_dates)} dates, {len(all_bus_dates)} valid bus-date cells total")

model_prior_weight=0.600, run_id=68
30 dates, 41332 valid bus-date cells total


In [5]:
def rows_for_date(trip_date: datetime.date) -> list[tuple]:
    """Solve one date, returning COPY-ready rows for every valid bus-date that day."""
    day_bus_ids = set(all_bus_dates.loc[all_bus_dates["date"] == trip_date, "bus_id"])
    cand_today = candidates_all[candidates_all["date"] == trip_date]
    cand_today = cand_today[cand_today["bus_id"].isin(day_bus_ids)]

    rows: list[tuple] = []
    no_candidate_buses = day_bus_ids - set(cand_today["bus_id"].unique())
    rows.extend(
        (b, trip_date, None, None, None, "no_candidates") for b in no_candidate_buses
    )

    if cand_today.empty:
        return rows

    votes_today = (
        votes_all[votes_all["date"] == trip_date] if not votes_all.empty else votes_all
    )
    raw = belief.compute_raw_beliefs(cand_today, votes_today, model_prior_weight)
    result = assignment.solve_date(raw, compute_margins=True)
    for bus_id, device_id, cost, margin in zip(
        result.bus_id, result.device_id, result.cost, result.margin, strict=True
    ):
        margin_val = None if np.isnan(margin) else float(margin)
        rows.append(
            (bus_id, trip_date, device_id, float(cost), margin_val, "global_assignment")
        )
    return rows

In [6]:
start = time.monotonic()
total_rows = 0
with (
    conn.cursor() as cur,
    cur.copy(
        "COPY ml.bus_matching_global_assignment "
        "(bus_id, date, device_id, cost, margin, method) FROM STDIN"
    ) as copy,
):
    for trip_date in all_dates:
        day_rows = rows_for_date(trip_date)
        for row in day_rows:
            copy.write_row(row)
        total_rows += len(day_rows)
        elapsed = time.monotonic() - start
        print(f"{trip_date}: {len(day_rows)} rows ({elapsed:.1f}s elapsed)")
conn.commit()
print(f"loaded {total_rows} rows in {time.monotonic() - start:.1f}s")

2023-11-01: 1652 rows (3.6s elapsed)


2023-11-02: 660 rows (4.0s elapsed)


2023-11-03: 1587 rows (7.3s elapsed)


2023-11-04: 1140 rows (8.6s elapsed)


2023-11-05: 612 rows (8.9s elapsed)


...

2023-11-26: 583 rows (65.9s elapsed)


2023-11-27: 1665 rows (69.7s elapsed)


2023-11-28: 1660 rows (73.3s elapsed)


2023-11-29: 1657 rows (77.0s elapsed)


2023-11-30: 1654 rows (80.7s elapsed)
loaded 41332 rows in 80.7s


## Sanity checks

In [7]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT count(*), count(DISTINCT bus_id || date::text) "
        "FROM ml.bus_matching_global_assignment;"
    )
    print("total rows / distinct bus-dates:", cur.fetchone())

    cur.execute(
        "SELECT method, count(*), count(*) FILTER (WHERE device_id IS NOT NULL) "
        "FROM ml.bus_matching_global_assignment GROUP BY method;"
    )
    print("method / total / with device:")
    for row in cur.fetchall():
        print(" ", row)

    dupe_sql = (
        "SELECT count(*) FROM (\n"
        "    SELECT date, device_id, count(DISTINCT bus_id) AS n_buses\n"
        "    FROM ml.bus_matching_global_assignment\n"
        "    WHERE device_id IS NOT NULL\n"
        "    GROUP BY date, device_id\n"
        "    HAVING count(DISTINCT bus_id) > 1\n"
        ") dupes;"
    )
    cur.execute(dupe_sql)
    print("device assigned to >1 bus on the same date (must be 0):", cur.fetchone())

    cur.execute(
        "SELECT count(*) FROM ml.bus_matching_global_assignment "
        "WHERE method = %(method)s AND device_id IS NULL;",
        {"method": "global_assignment"},
    )
    print(
        "solved buses assigned 'none' (real candidates existed, none good enough):",
        cur.fetchone(),
    )

    cur.execute(
        "SELECT avg(margin), percentile_cont(0.5) WITHIN GROUP (ORDER BY margin) "
        "FROM ml.bus_matching_global_assignment WHERE margin IS NOT NULL;"
    )
    print("mean / median margin:", cur.fetchone())

total rows / distinct bus-dates: (41332, 41332)
method / total / with device:
  ('global_assignment', 40007, 30121)
  ('no_candidates', 1325, 0)
device assigned to >1 bus on the same date (must be 0): (0,)
solved buses assigned 'none' (real candidates existed, none good enough): (9886,)
mean / median margin: (2.2303515270781293, 2.515688099515387)


In [8]:
sample_sql = (
    "SELECT bus_id, date, device_id, cost, margin, method\n"
    "FROM ml.bus_matching_global_assignment\n"
    "WHERE device_id IS NOT NULL\n"
    "ORDER BY margin ASC NULLS LAST\n"
    "LIMIT 15;"
)
sample = pd.read_sql(sample_sql, conn)
print("15 lowest-margin (most contested) assignments:")
sample

15 lowest-margin (most contested) assignments:


/tmp/ipykernel_1023330/1215165295.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample = pd.read_sql(sample_sql, conn)


,bus_id,date,device_id,cost,margin,method
0,35132,2023-11-02,ep1-428103584,1.041962,-1.136868e-13,global_assignment
1,30708,2023-11-02,ep1-428103715,1.041962,-1.136868e-13,global_assignment
2,30708,2023-11-03,ep1-428103715,1.041962,3.410605e-13,global_assignment
3,21724,2023-11-21,ep1-540675373,0.160853,3.410605e-13,global_assignment
4,35726,2023-11-09,ep1-428113883,0.160853,6.821210e-13,global_assignment
5,30708,2023-11-11,ep1-428103715,1.041962,1.762146e-12,global_assignment
6,14303,2023-11-04,ep1-428110689,0.152843,1.875833e-12,global_assignment
7,36315,2023-11-18,ep1-428110248,0.160853,2.557954e-12,global_assignment
8,36304,2023-11-05,ep1-428109189,1.608686,2.746768e-04,global_assignment
9,14002,2023-11-28,ep1-428108975,1.112731,4.888163e-04,global_assignment
